In [14]:
import pandas as pd
from Levenshtein import distance

tm = pd.read_csv("../data/scrape/pro/player_stats.csv")
tm_matches = pd.read_csv("../data/scrape/pro/matches.csv")
tm_players = pd.read_csv("../data/scrape/pro/players.csv")

tm_players = tm_players[["player_id", "player_name", "position"]]
tm_matches = tm_matches[["match_id", "date", "home_club_id", "away_club_id", "home_goals", "away_goals"]]

tm = tm.merge(tm_players, on="player_id", how="left")
tm = tm.merge(tm_matches, on="match_id", how="left")

ss = pd.read_csv("../data/scrape/pro/ratings.csv")
ss = ss.rename(columns={"datum": "date"})
ss = ss.rename(columns={"name": "player_name"})

In [ ]:
def get_result(row):
    if row["club_id"] == row["home_club_id"]:
        if row["home_goals"] > row["away_goals"]:
            return "win"
        elif row["home_goals"] < row["away_goals"]:
            return "loss"
        else:
            return "draw"
    
    elif row["club_id"] == row["away_club_id"]:
        if row["away_goals"] > row["home_goals"]:
            return "win"
        elif row["away_goals"] < row["home_goals"]:
            return "loss"
        else:
            return "draw"
    
    else:
        return None

tm["result"] = tm.apply(get_result, axis=1)



ss["player_name"] = (
    ss["player_name"]
    .str.replace(r"\d+", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

ss["player_name"] = (
    ss["player_name"]
    .str.replace(r"Basel|FC Lugano|FC Sion", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

ss["player_name"] = ss["player_name"].str.replace(".", "", regex=False).str.strip()
ss["player_name"] = ss["player_name"].str.lower()
tm["player_name"] = tm["player_name"].str.lower()
tm = tm.dropna(subset=["player_name"])

ss.head()
tm.head()

,player_id,match_id,club_id,goals,assists,yellow,yellow_red,red,start_eleven,minutes,...,team_goals,team_conceded,player_name,position,date,home_club_id,away_club_id,home_goals,away_goals,result
0,448169,4361914,2790,0,0,0,0,0,1,90,...,2,1,amir saipi,Torwart,2024-07-20,2790,504,2.0,1.0,win
1,345780,4361914,2790,0,0,1,0,0,1,68,...,1,1,lukas mai,Innenverteidiger,2024-07-20,2790,504,2.0,1.0,win
2,860052,4361914,2790,0,0,0,0,0,1,90,...,2,1,ayman el wafi,Innenverteidiger,2024-07-20,2790,504,2.0,1.0,win
3,422401,4361914,2790,0,0,0,0,0,1,90,...,2,1,milton valenzuela,Linker Verteidiger,2024-07-20,2790,504,2.0,1.0,win
4,395676,4361914,2790,0,0,0,0,0,1,90,...,2,1,zachary brault-guillard,Rechter Verteidiger,2024-07-20,2790,504,2.0,1.0,win


In [10]:
def merge_and_split(tm, ss, df_matched, on_cols):
    # 1. Exakter Merge
    merged = tm.merge(
        ss,
        on=on_cols,
        how="left",
        indicator=True
    )

    exact_matches = merged[merged["_merge"] == "both"].drop(columns="_merge")
    n_exact = len(exact_matches)

    df_matched = pd.concat([df_matched, exact_matches], ignore_index=True)

    tm_remaining = tm.merge(
        exact_matches[on_cols].drop_duplicates(),
        on=on_cols,
        how="left",
        indicator=True
    )
    tm_remaining = tm_remaining[tm_remaining["_merge"] == "left_only"].drop(columns="_merge")

    # 2. Fuzzy-Match nur fuer die uebrigen
    if "player_name" not in on_cols:
        print(f"{n_exact} neue Spieler gematcht")
        return tm_remaining, df_matched

    other_cols = [c for c in on_cols if c != "player_name"]

    fuzzy_rows = []
    matched_tm_idx = set()
    matched_ss_idx = set()

    for tm_idx, tm_row in tm_remaining.iterrows():
        candidates = ss.copy()

        for col in other_cols:
            candidates = candidates[candidates[col] == tm_row[col]]

        for ss_idx, ss_row in candidates.iterrows():
            if ss_idx in matched_ss_idx:
                continue

            if distance(str(tm_row["player_name"]), str(ss_row["player_name"])) == 1:
                matched_tm_idx.add(tm_idx)
                matched_ss_idx.add(ss_idx)

                row = {**tm_row.to_dict(), **ss_row.to_dict()}
                fuzzy_rows.append(row)
                break

    fuzzy_matches = pd.DataFrame(fuzzy_rows)
    n_fuzzy = len(fuzzy_matches)

    if not fuzzy_matches.empty:
        df_matched = pd.concat([df_matched, fuzzy_matches], ignore_index=True)

    tm_new = tm_remaining.drop(index=list(matched_tm_idx)).reset_index(drop=True)

    print(f"{n_exact} exakte + {n_fuzzy} fuzzy Matches (Levenshtein = 1)")

    return tm_new, df_matched

In [11]:
df = pd.DataFrame()
tf,df = merge_and_split(tm, ss, df, ["player_name", "date"])

9013 exakte + 1147 fuzzy Matches (Levenshtein = 1)


In [12]:
ss.loc[ss["player_name"] == "lars lukas mai", "player_name"] = "lukas mai"
ss.loc[ss["player_name"] == "anto grgić", "player_name"] = "anto grgic"
ss.loc[ss["player_name"] == "daniel dos santos correia", "player_name"] = "daniel dos santos"
ss.loc[ss["player_name"] == "théo ndicka matam", "player_name"] = "théo ndicka"
ss.loc[ss["player_name"] == "tsiy william ndenge", "player_name"] = "tsiy ndenge"
ss.loc[ss["player_name"] == "filipe de carvalho ferreira", "player_name"] = "filipe de carvalho"
ss.loc[ss["player_name"] == "evans fabrice maurin", "player_name"] = "evans maurin"
ss.loc[ss["player_name"] == "kevin carlos omoruyi", "player_name"] = "kevin carlos"
ss.loc[ss["player_name"] == "nemanja tošić", "player_name"] = "nemanja tosic"
ss.loc[ss["player_name"] == "lawrence ati", "player_name"] = "lawrence ati zigi"
ss.loc[ss["player_name"] == "miguel changa chaiwa", "player_name"] = "miguel chaiwa"
ss.loc[ss["player_name"] == "łukasz łakomy", "player_name"] = "lukasz lakomy"
ss.loc[ss["player_name"] == "baltazar costa", "player_name"] = "baltazar"
ss.loc[ss["player_name"] == "dejan đokić", "player_name"] = "dejan djokic"
ss.loc[ss["player_name"] == "mamadou kaly sene", "player_name"] = "kaly sène"
ss.loc[ss["player_name"] == "jonas adjei adjetey", "player_name"] = "jonas adjetey"
ss.loc[ss["player_name"] == "adrian leon barišić", "player_name"] = "adrian leon barisic"
ss.loc[ss["player_name"] == "stefan knežević", "player_name"] = "stefan knezevic"
ss.loc[ss["player_name"] == "liam scott chipperfield", "player_name"] = "liam chipperfield"
ss.loc[ss["player_name"] == "marin šotiček", "player_name"] = "marin soticek"
ss.loc[ss["player_name"] == "mamadou usman simbakoli", "player_name"] = "usman simbakoli"
ss.loc[ss["player_name"] == "jovan milošević", "player_name"] = "jovan milosevic"
ss.loc[ss["player_name"] == "s lauper", "player_name"] = "sandro lauper"
ss.loc[ss["player_name"] == "abdou karim sow", "player_name"] = "karim sow"
ss.loc[ss["player_name"] == "jeremy frick", "player_name"] = "jérémy frick"
ss.loc[ss["player_name"] == "mohssine bourika", "player_name"] = "mouhcine bouriga"
ss.loc[ss["player_name"] == "josafat mendes", "player_name"] = "joe mendes"
ss.loc[ss["player_name"] == "a sanches", "player_name"] = "alvyn sanches"
ss.loc[ss["player_name"] == "ricardo azevedo alves", "player_name"] = "ricardo alves"
ss.loc[ss["player_name"] == "a sanches", "player_name"] = "alvyn sanches"
ss.loc[ss["player_name"] == "mateusz łęgowski", "player_name"] = "mateusz legowski"
ss.loc[ss["player_name"] == "s kapino", "player_name"] = "stefanos kapino"
ss.loc[ss["player_name"] == "musa araz", "player_name"] = "musa araz"
ss.loc[ss["player_name"] == "tomas veron lupi", "player_name"] = "tomás verón lupi"
ss.loc[ss["player_name"] == "joseph belmar", "player_name"] = "belmar joseph"
ss.loc[ss["player_name"] == "n burkart", "player_name"] = "nishan burkart"
ss.loc[ss["player_name"] == "felix tsimba", "player_name"] = "emmanuel tsimba"
ss.loc[ss["player_name"] == "hassane imourane", "player_name"] = "imourane hassane"
ss.loc[ss["player_name"] == "rúben dantas", "player_name"] = "rúben dantas fernandes"
ss.loc[ss["player_name"] == "k hajrizi", "player_name"] = "kreshnik hajrizi"
ss.loc[ss["player_name"] == "paul bernardoni", "player_name"] = "paul bernardoni"
ss.loc[ss["player_name"] == "t golliard", "player_name"] = "théo golliard"
ss.loc[ss["player_name"] == "vasilije janjičić", "player_name"] = "vasilije janjicic"
ss.loc[ss["player_name"] == "m käit", "player_name"] = "mattias käit"
ss.loc[ss["player_name"] == "thelonius bair", "player_name"] = "theo bair"
ss.loc[ss["player_name"] == "b labeau", "player_name"] = "brighton labeau"
ss.loc[ss["player_name"] == "papa souleymane n'diaye", "player_name"] = "souleymane n'diaye"
ss.loc[ss["player_name"] == "g montolio", "player_name"] = "genís montolio"
ss.loc[ss["player_name"] == "d pech", "player_name"] = "dominik pech"
ss.loc[ss["player_name"] == "miguel mardochee", "player_name"] = "mardochée miguel"

tf,df = merge_and_split(tm, ss, df, ["player_name", "date"])

df = df.dropna(subset=["rating"])

10370 exakte + 1082 fuzzy Matches (Levenshtein = 1)


In [13]:
df.to_csv("../data/transform/pro/stats_with_rating.csv", index=False)

In [26]:
tf

,player_id,match_id,club_id,goals,assists,yellow,yellow_red,red,start_eleven,minutes,on_min,off_min,team_goals,team_conceded,player_name,position,date
